# CoPoLLM：Colab 推理与公开子集评测起步
以 Qwen3-8B 为例，对比作者 LoRA 与同量化基座。**不是完整训练流水线，也不是论文原表复现。**
已在本地核查数据与 Python 语法；尚未在 Colab GPU 实测。先选 GPU，按顺序运行。
完整说明见同目录《CoPoLLM-Colab复现方案.md》。不要把缺失 annotation 当成无偏差。

Qwen3 专项说明：本文件保留仓库自定义任务提示格式，并未调用 apply_chat_template。enable_thinking=False 是 chat template 的参数，不能直接加到 model.generate 中。仓库提示的 no think 是文本要求，不能视为硬关闭思考模式。若输出出现 <think>，先记录并核对官方推理模板，不在测试集上反复挑选提示来追分。


In [ ]:
%pip install -q 'transformers==4.51.3' 'peft==0.15.2' 'accelerate==1.6.0' 'bitsandbytes==0.49.0' 'huggingface_hub==0.30.2' scikit-learn
# 不替换 Colab 的 torch。安装后重启运行时，再从下一格继续；不要再次运行本安装格。


## 挂载 Google Drive（可使用另一个账号）

下方代码会将实验结果保存到所授权账号的 `MyDrive/CoPoLLM/public_subset_eval`。打开 Colab 的账号和用于保存结果的 Drive 账号可以不同。

1. 运行下一格，按提示授权；出现账号选择窗口时，选择希望保存结果的 Google 账号。
2. 授权完成后，查看输出的 `MyDrive` 顶层文件名，并与目标账号中的内容核对。文件列表只能辅助确认，不会直接显示账号身份；需要时在 Google Drive 网页中核对账号。
3. 若已挂载错账号，在**确认文件已保存且没有写入进行中**后，单独运行下面的代码卸载，再重新运行下一格并选择目标账号：

```python
from google.colab import drive
drive.flush_and_unmount()
```

如果授权窗口自动选错账号，可尝试在窗口中切换账号，或使用浏览器无痕窗口重新打开 Colab。挂载后，此 Notebook 的代码可访问所授权 Drive 中的文件，请只运行可信代码。完整评测建议保留 Drive 保存，并将重要结果另行下载备份。


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, json, sys, platform, time, hashlib, random, ast, re
import numpy as np
import torch
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sklearn.metrics import precision_recall_fscore_support
from contextlib import nullcontext

drive.mount('/content/drive')
print('请核对以下 MyDrive 内容是否属于目标账号：')
print([p.name for p in Path('/content/drive/MyDrive').iterdir()][:30])
OUT = Path('/content/drive/MyDrive/CoPoLLM/public_subset_eval')
OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), '请在 Colab 运行时设置中选择 GPU'
print(subprocess.check_output(['nvidia-smi'], text=True))
print('torch', torch.__version__, 'CUDA', torch.version.cuda)
random.seed(42); np.random.seed(42); torch.manual_seed(42)
# 首次 10 条，再改为 None 跑全部。不同实验条件必须使用新的 RUN_NAME。
LIMIT = 10
RUN_NAME = 'qwen3_8b_nf4_ctx2048_v1'
MAX_INPUT = 2048
MAX_NEW = 128

In [ ]:
REPO = Path('/content/CoPoLLM-for-ACL-2026')
COMMIT = 'cfe92bb0228da702af910adf31b1a64a5e13cb8b'
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/Chips98/CoPoLLM-for-ACL-2026.git',str(REPO)], check=True)
subprocess.run(['git','checkout',COMMIT],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip() == COMMIT
train = json.loads((REPO/'data/train.json').read_text())
test = json.loads((REPO/'data/test.json').read_text())
print('train dialogues',len(train),'test dialogues',len(test))
# 只解析源码中的常量，不 import 训练脚本，不执行论文或数据中的指令。
needed = {'CLS_TASK_ID','GEN_TASK_ID','CLS_MARKER','GEN_MARKER',
          'RESPONSE_TEMPLATE','SYS_PROMPT_CLASSIFY','SYS_PROMPT_GENERATION'}
constants = {}
for node in ast.parse((REPO/'task4/sft_unified.py').read_text()).body:
    if isinstance(node,ast.Assign):
        for target in node.targets:
            if isinstance(target,ast.Name) and target.id in needed:
                constants[target.id] = ast.literal_eval(node.value)
assert set(constants) == needed
print('已读取源码任务模板')

In [ ]:
LABELS = ['非黑即白','过度概括','灾难化','读心术','情感推理','应该句式','标签化','个人化']
VALID_RISKS = {'低危','中危','高危','无'}
VALID_INTENSITIES = {'轻微','中等','严重','无'}
def normalize_tags(value):
    if isinstance(value,str):
        return [x.strip() for x in re.split(r'[、,，;；/|]+',value) if x.strip()]
    if isinstance(value,list):
        return [str(x).strip() for x in value if str(x).strip()]
    return []

def make_prompt(task, history, user):
    cls = task == 'classify'
    task_id = constants['CLS_TASK_ID' if cls else 'GEN_TASK_ID']
    marker = constants['CLS_MARKER' if cls else 'GEN_MARKER']
    system = constants['SYS_PROMPT_CLASSIFY' if cls else 'SYS_PROMPT_GENERATION']
    history_text = ''.join(('患者：' if m['role']=='user' else '治疗师：')+m['content']+'\n' for m in history)
    return f"{task_id}\n{marker}\n{system}\n历史对话：\n{history_text}患者：{user}" + constants['RESPONSE_TEMPLATE']

samples=[]
excluded=0
for di,conv in enumerate(test):
    history=[]
    for ti,msg in enumerate(conv['messages']):
        if msg.get('role')=='user' and isinstance(msg.get('annotation'),dict):
            a=msg['annotation']; tags=normalize_tags(a.get('bias_tags'))
            valid = bool(tags) and all(x in LABELS for x in tags) and a.get('risk_level') in VALID_RISKS and a.get('bias_intensity') in VALID_INTENSITIES
            if valid:
                samples.append(dict(key=f'{di}:{ti}',history=list(history),user=msg['content'],
                                    gold_tags=tags,gold_risk=a['risk_level'],gold_intensity=a['bias_intensity']))
            else:
                excluded+=1
        if msg.get('role') in ('user','assistant'):
            history.append({'role':msg['role'],'content':msg['content']})
print('有效诊断样本',len(samples),'排除无效标注',excluded,
      '高危',sum(x['gold_risk']=='高危' for x in samples))
assert len(samples)==981 and sum(x['gold_risk']=='高危' for x in samples)==42
(OUT/'sample_manifest.json').write_text(json.dumps([{k:v for k,v in s.items() if k not in ('history','user')} for s in samples],ensure_ascii=False,indent=2))

In [ ]:
BASE = 'Qwen/Qwen3-8B'
ADAPTER_REPO = 'Chips95/Lora_Adapter_for_ACL2026_CoPoLLM'
SUBDIR = 'lora_adapter_for_qwen3-8b'
api = HfApi()
# 首次解析后写入锁文件；后续续跑沿用相同 revisions。
lock_path=OUT/(RUN_NAME+'_revisions.json')
if lock_path.exists():
    lock=json.loads(lock_path.read_text())
else:
    lock={'base':api.model_info(BASE).sha,'adapter':api.model_info(ADAPTER_REPO).sha}
    lock_path.write_text(json.dumps(lock,indent=2))
local = Path(snapshot_download(ADAPTER_REPO,revision=lock['adapter'],allow_patterns=[
    SUBDIR+'/adapter_config.json',SUBDIR+'/adapter_model.safetensors',
    SUBDIR+'/tokenizer*',SUBDIR+'/special_tokens_map.json',SUBDIR+'/added_tokens.json',
    SUBDIR+'/vocab.json',SUBDIR+'/merges.txt']))/SUBDIR
config=json.loads((local/'adapter_config.json').read_text())
print('作者基座路径',config['base_model_name_or_path'],'实际使用',BASE)
tokenizer=AutoTokenizer.from_pretrained(local)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token=tokenizer.eos_token
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
base=AutoModelForCausalLM.from_pretrained(BASE,revision=lock['base'],
    quantization_config=quant,torch_dtype=torch.float16,device_map={'':0},attn_implementation='sdpa')
assert len(tokenizer)<=base.get_input_embeddings().num_embeddings, '词表不匹配，先核查 tokenizer，勿盲目 resize'
model=PeftModel.from_pretrained(base,str(local))
model.eval()
(OUT/(RUN_NAME+'_pip-freeze.txt')).write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
manifest={'commit':COMMIT,'base':BASE,'adapter_repo':ADAPTER_REPO,'subdir':SUBDIR,'revisions':lock,
          'max_input':MAX_INPUT,'max_new':MAX_NEW,'torch':torch.__version__,'cuda':torch.version.cuda,
          'gpu':torch.cuda.get_device_name(0),'quantization':'NF4 double quant FP16',
          'template_sha256':hashlib.sha256(json.dumps(constants,ensure_ascii=False,sort_keys=True).encode()).hexdigest()}
manifest_path=OUT/(RUN_NAME+'_manifest.json')
if manifest_path.exists():
    old=json.loads(manifest_path.read_text())
    for k in ('commit','base','adapter_repo','subdir','revisions','max_input','max_new','quantization','template_sha256'):
        assert old[k]==manifest[k], f'实验条件改变：{k}，请换 RUN_NAME'
else:
    manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2))

In [ ]:
def fit_prompt(sample,task):
    history=list(sample['history']); dropped=0
    while True:
        prompt=make_prompt(task,history,sample['user'])
        inputs=tokenizer(prompt,return_tensors='pt',add_special_tokens=False)
        if inputs['input_ids'].shape[1]<=MAX_INPUT:
            return inputs,dropped
        if not history:
            raise ValueError('当前发言+任务指令已超长；提高 MAX_INPUT 并更换 RUN_NAME')
        # 按轮次删除最早历史，保留任务指令、当前发言和响应锚点。
        n=1
        if history[0]['role']=='user' and len(history)>1 and history[1]['role']=='assistant':
            n=2
        history=history[n:]; dropped+=n

@torch.inference_mode()
def predict(sample, task='classify', use_adapter=True):
    inputs,dropped=fit_prompt(sample,task)
    inputs={k:v.to('cuda:0') for k,v in inputs.items()}
    n=inputs['input_ids'].shape[1]
    ctx=nullcontext() if use_adapter else model.disable_adapter()
    with ctx:
        out=model.generate(**inputs,do_sample=False,max_new_tokens=MAX_NEW,
                           pad_token_id=tokenizer.pad_token_id,eos_token_id=tokenizer.eos_token_id)
    text=tokenizer.decode(out[0,n:],skip_special_tokens=True)
    return {'text':text,'input_tokens':n,'dropped_history_messages':dropped,
            'hit_generation_limit':out.shape[1]-n>=MAX_NEW}

# 当前 target 回复与 annotation 均未进入提示。
print(predict(samples[0]))
print(predict(samples[0],task='generation'))

先检查上面两条输出。分类应有认知偏差标签、安全风险、偏差强度；生成应为自然回复。
分类模板保留仓库自定义任务头。基座使用同样模板进行受控对照；它不代表论文基座的官方提示设置。
确认后，先 LIMIT=10 跑通；再回到参数格改为 LIMIT=None，只重跑参数与本格之后即可。

In [ ]:
selected=samples if LIMIT is None else samples[:LIMIT]
results_path=OUT/(RUN_NAME+'_diagnosis.jsonl')
existing=[]
if results_path.exists():
    for line in results_path.read_text().splitlines():
        if line.strip(): existing.append(json.loads(line))
done={(x['key'],x['condition']) for x in existing}
started=time.time(); count=0
with results_path.open('a',encoding='utf-8') as f:
    for s in selected:
        for condition,use_adapter in [('base',False),('copollm',True)]:
            if (s['key'],condition) in done: continue
            prediction=predict(s,use_adapter=use_adapter)
            row={k:s[k] for k in ('key','gold_tags','gold_risk','gold_intensity')}
            row.update(condition=condition,**prediction)
            f.write(json.dumps(row,ensure_ascii=False)+'\n'); f.flush()
            done.add((s['key'],condition)); count+=1
            if count%10==0: print('本次完成',count,'预测，秒数',round(time.time()-started,1))
print('结果',results_path)

In [ ]:
def parse_prediction(text):
    fields={}
    for field in ('认知偏差标签','安全风险','偏差强度'):
        m=re.search(re.escape(field)+r'[：:]\s*([^\n]+)',text)
        fields[field]=m.group(1).strip() if m else None
    tags=normalize_tags(fields['认知偏差标签'])
    known=bool(tags) and all(x in LABELS or x=='无' for x in tags)
    pred_tags=[x for x in tags if x in LABELS]
    risk=fields['安全风险']; intensity=fields['偏差强度']
    return pred_tags,risk,intensity,not (known and risk in VALID_RISKS and intensity in VALID_INTENSITIES)

wanted={s['key'] for s in selected}
rows=[json.loads(l) for l in results_path.read_text().splitlines() if l.strip()]
report={}
for condition in ('base','copollm'):
    current=[x for x in rows if x['condition']==condition and x['key'] in wanted]
    assert len(current)==len(selected), '预测未完成，请继续运行前一格'
    parsed=[parse_prediction(x['text']) for x in current]
    gold=np.array([[int(t in x['gold_tags']) for t in LABELS] for x in current])
    pred=np.array([[int(t in p[0]) for t in LABELS] for p in parsed])
    macro=precision_recall_fscore_support(gold,pred,average='macro',zero_division=0)
    micro=precision_recall_fscore_support(gold,pred,average='micro',zero_division=0)
    risks=[i for i,x in enumerate(current) if x['gold_risk']=='高危']
    missed=sum(parsed[i][1]!='高危' for i in risks)
    per=precision_recall_fscore_support(gold,pred,average=None,zero_division=0)
    report[condition]={'n':len(current),'macro_precision':float(macro[0]),'macro_recall':float(macro[1]),
        'macro_f1':float(macro[2]),'micro_f1':float(micro[2]),'high_risk_n':len(risks),
        'HRMDR':missed/len(risks) if risks else None,
        'parse_failure_rate':sum(p[3] for p in parsed)/len(parsed),
        'input_trim_rate':sum(x['dropped_history_messages']>0 for x in current)/len(current),
        'generation_limit_rate':sum(x['hit_generation_limit'] for x in current)/len(current),
        'per_label':{t:{'precision':float(per[0][i]),'recall':float(per[1][i]),'f1':float(per[2][i]),'support':int(per[3][i])} for i,t in enumerate(LABELS)}}
print(json.dumps(report,ensure_ascii=False,indent=2))
(OUT/(RUN_NAME+'_metrics.json')).write_text(json.dumps(report,ensure_ascii=False,indent=2))

## 接下来怎么做
- 全部 981 条完成后，结果标为“公开子集、4-bit、2048 上下文、仓库训练模板的近似评测”。不能当作原表 3 的重现。
- 本 Notebook 还没有生成质量自动打分、专家评分或任何微调训练。可对固定生成样本调用 `predict(s, task='generation')`，逐条保存后依论文提示盲评。
- CDD 固定八类做 macro average。当前有效公开样本金标只有单类别；无法完整验证论文多标签能力。
- LIMIT=10 时许多类和高危样本可能缺失，分数仅用于调试。完整子集 HRMDR 的分母只有 42。
- 缺失/空 annotation 不进入评测，不等于阴性。原始输出已保存，解析失败和上下文裁剪单独报告。
- 开始训练前请按方案修复数据 schema、next-state、任务权重与 max_steps 等问题。

来源：[论文](https://arxiv.org/html/2604.17178v1)、[代码](https://github.com/Chips98/CoPoLLM-for-ACL-2026)、[作者权重](https://huggingface.co/Chips95/Lora_Adapter_for_ACL2026_CoPoLLM/tree/main)、[Colab FAQ](https://research.google.com/colaboratory/faq.html)。